# Part 1
Hallee Pham

## Setup
* Imports
* Read data

In [1]:
import pandas as pd
import numpy as np

# Read data
cars = pd.read_csv('https://raw.githubusercontent.com/halleepham/DataScience-Assignment2/refs/heads/main/Part1/data/raw/train.csv', index_col=0)
cars.head()

,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


## Part B
> Note: I did this part before removing missing values (Part A)
* Remove units from attributes and only keep numerical values

In [2]:
# columns that have units
unit_columns = ['Mileage', 'Engine', 'Power', 'New_Price']

# keep only numerical values and converting to float type
for col in unit_columns:
  cars[col] = cars[col].str.extract(r'([\d.]+)').astype(float)

cars.head(5)

,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67,1582.0,126.20,5.0,NaN,12.50
2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13.00,1199.0,88.70,5.0,8.61,4.50
3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77,1248.0,88.76,7.0,NaN,6.00
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.20,1968.0,140.80,5.0,NaN,17.74
6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08,1461.0,63.10,5.0,NaN,3.50


## Part A
* Look for missing values
* Replace or drop them and justify choice.

In [3]:
# Check which columns have missing values
na_summary = pd.DataFrame({
    'missing_count': cars.isna().sum(),
    'missing_percentage': round(cars.isna().sum() / len(cars) * 100, 4)
})
na_summary

,missing_count,missing_percentage
Name,0,0.0000
Location,0,0.0000
Year,0,0.0000
Kilometers_Driven,0,0.0000
Fuel_Type,0,0.0000
Transmission,0,0.0000
Owner_Type,0,0.0000
Mileage,2,0.0342
Engine,36,0.6157
Power,36,0.6157


Observations:  
* `Mileage`, `Engine`, `Power`, and `Seats` have fairly low number of missing values
* `New_Price` has over 86% of values missing. I will drop this column.   

Next, I check the skewness of each column with missing values to determine course of action.

In [4]:
na_columns = ['Mileage', 'Engine', 'Power', 'Seats']

# Print skewness values of columns with missing values
for col in na_columns:
  print(f'{col}: Skewness = {cars[col].skew():.2f}')

# Since Seats is a discrete variable, view count distribution
print(f'\n{cars['Seats'].value_counts().sort_index()}')

Mileage: Skewness = -0.31
Engine: Skewness = 1.41
Power: Skewness = 1.92
Seats: Skewness = 1.91

Seats
2.0       13
4.0       93
5.0     4866
6.0       29
7.0      668
8.0      133
9.0        3
10.0       4
Name: count, dtype: int64


Observations:  
* `Mileage` is approximatley symmetric
* The other columns are highly positively skewed
* The most frequent number of seats is 5

In [5]:
# Replace missing values in Mileage with mean
cars['Mileage'] = cars['Mileage'].fillna(cars['Mileage'].mean())

# Replace missing values in Engine with median (rounded)
cars['Engine'] = cars['Engine'].fillna(round(cars['Engine'].median(), 0))

# Replace missing values in Power with median
cars['Power'] = cars['Power'].fillna(cars['Power'].median())

# Replace missing values in Seats with mode
cars['Seats'] = cars['Seats'].fillna(cars['Seats'].mode()[0])

# Drop the New_Price column
cars.drop('New_Price', axis=1, inplace=True)

# Change Engine and Seats to integers (from floats)
cars['Engine'] = cars['Engine'].astype(int)
cars['Seats'] = cars['Seats'].astype(int)

cars.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5847 entries, 1 to 6018
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               5847 non-null   object 
 1   Location           5847 non-null   object 
 2   Year               5847 non-null   int64  
 3   Kilometers_Driven  5847 non-null   int64  
 4   Fuel_Type          5847 non-null   object 
 5   Transmission       5847 non-null   object 
 6   Owner_Type         5847 non-null   object 
 7   Mileage            5847 non-null   float64
 8   Engine             5847 non-null   int64  
 9   Power              5847 non-null   float64
 10  Seats              5847 non-null   int64  
 11  Price              5847 non-null   float64
dtypes: float64(3), int64(4), object(5)
memory usage: 593.8+ KB


### Justifications for Replacement/Drop
* `Mileage`: missing values were replaced with the **mean**, as the skewness value (-0.31) shows that the distribution is fairly symmetric. Median could also have been chosen, but since the distribution is symmetric, using the mean will preserve the original mean of the column after imputation. Also, only 2 missing values were present so using mean over median may not have any significant effects.
* `Engine`: missing values were replaced with **median**. The skewness value (1.41) is greater than 1, indicating a highly right-skewed distribution. In this case, median is better than mean because mean is heavily sensitive to outliers. Median better represents a typical value in the skewed data, which is the goal of imputation. I also decided to round the median value before replacement, because by visual inspection of the data and basic domain knowledge, engine displacement values are usually whole numbers.
* `Power`: missing values were replaced with **median** for the same reason as `Engine`. The only difference is that the median value is not rounded since power values are typically decimals.
* `Seats`: missing values were replaced with **mode**. While it technically is numerical data, this variable is discrete and categorical-like. There is only a small fixed set of possible values. The best assumption we can make is that cars with the missing values are probably similar to the majority of cars in the dataset. Also with basic knowledge, I know that a majority of cars have 5 seats anyways, and this is supported by the value counts. Mean or median could give a decimal value, which would not provide a real valid value.
* `New_Price`: This column was **dropped** from the dataset. ~86% of the values were missing in this column, so if I decided to impute missing values, I'd essentially be making the data up. In a real situation, the missing values of this column could be searched up online.

## Part C
* Change categorical variables into numerical one hot encoded values

In [6]:
# Encode Fuel_Type and Transmission, and convert to integers (from boolean)
cars_encoded = pd.get_dummies(cars, columns=['Fuel_Type', 'Transmission'], dtype=int)

cars_encoded.head()

,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,First,19.67,1582,126.20,5,12.50,1,0,0,0,1
2,Honda Jazz V,Chennai,2011,46000,First,13.00,1199,88.70,5,4.50,0,0,1,0,1
3,Maruti Ertiga VDI,Chennai,2012,87000,First,20.77,1248,88.76,7,6.00,1,0,0,0,1
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Second,15.20,1968,140.80,5,17.74,1,0,0,1,0
6,Nissan Micra Diesel XV,Jaipur,2013,86999,First,23.08,1461,63.10,5,3.50,1,0,0,0,1


## Part D
* Add one feature and add to the dataset
* I added a column `Car_Age` that represents the current age of the car

In [7]:
from datetime import datetime

cars_final = cars_encoded.copy()
cars_final['Car_Age'] = datetime.now().year - cars_final['Year']

cars_final.head()

,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,Price,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual,Car_Age
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,First,19.67,1582,126.20,5,12.50,1,0,0,0,1,11
2,Honda Jazz V,Chennai,2011,46000,First,13.00,1199,88.70,5,4.50,0,0,1,0,1,15
3,Maruti Ertiga VDI,Chennai,2012,87000,First,20.77,1248,88.76,7,6.00,1,0,0,0,1,14
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Second,15.20,1968,140.80,5,17.74,1,0,0,1,0,13
6,Nissan Micra Diesel XV,Jaipur,2013,86999,First,23.08,1461,63.10,5,3.50,1,0,0,0,1,13


## Part E


### Select and Filter Operations
I used these operations to answer the question "What are the names and price of all first owner cars made after 2011?"

In [8]:
# Select and Filter operations
cars_final.loc[(cars_final['Owner_Type'] == 'First') & (cars_final['Year'] > 2011), ['Name', 'Price', 'Owner_Type', 'Year']]

,Name,Price,Owner_Type,Year
1,Hyundai Creta 1.6 CRDi SX Option,12.50,First,2015
3,Maruti Ertiga VDI,6.00,First,2012
6,Nissan Micra Diesel XV,3.50,First,2013
7,Toyota Innova Crysta 2.8 GX AT 8S,17.50,First,2016
8,Volkswagen Vento Diesel Comfortline,5.20,First,2013
...,...,...,...,...
6010,Honda Brio 1.2 VX MT,3.20,First,2013
6013,Honda Amaze VX i-DTEC,4.83,First,2015
6014,Maruti Swift VDI,4.75,First,2014
6015,Hyundai Xcent 1.1 CRDi S,4.00,First,2015


### Rename/Mutate Operations
* I renamed `Engine` to `Engine_Displacement` and `Power` to `Engine_Horsepower`
* I added a new column (mutate) `Engine_Efficiency`

In [9]:
# Rename operation
cars_final = cars_final.rename(columns={'Engine': 'Engine_Displacement', 'Power': 'Engine_Horsepower'})

# Mutate operation
cars_final['Engine_Efficiency'] = round(cars_final['Engine_Horsepower'] / cars_final['Engine_Displacement'], 4)

cars_final[['Engine_Displacement', 'Engine_Horsepower', 'Engine_Efficiency']].head()

,Engine_Displacement,Engine_Horsepower,Engine_Efficiency
1,1582,126.20,0.0798
2,1199,88.70,0.0740
3,1248,88.76,0.0711
4,1968,140.80,0.0715
6,1461,63.10,0.0432


### Arrange and Summarize with Groupby Operations
* I used a combination of these operations to answer the question " What is the average price and average kilometers driven by location, sorted by average price descending?"

In [10]:
cars_final.groupby('Location').agg(
  Avg_Price=('Price', 'mean'),
  Avg_KM=('Kilometers_Driven', 'mean')
).round(2).sort_values(by='Avg_Price', ascending=False)

,Avg_Price,Avg_KM
Location,,
Coimbatore,15.16,46949.78
Bangalore,13.48,58227.86
Kochi,11.31,44718.00
Hyderabad,10.00,70651.65
Delhi,9.88,57185.81
Mumbai,9.59,44666.16
Ahmedabad,8.57,54974.70
Chennai,7.96,90494.86
Pune,6.95,69785.16


In [11]:
# save final processed data set into a csv
cars_final.to_csv('cars_processed.csv', index=False)